In [1]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

### 메시지발송 시간, 재방문 여부에 따른 종속변수 코딩
* sess 이후 메시지 발송 이력이 없으면 -1
* sess 이후 메시지 발송 이력이 있고, 재방문을 했다면 1
* sess 이후 메시지 발송 이력이 있고, 재방문을 하지 않았다면 0
* 메시지 데이터 merge

In [2]:
sess = pd.read_csv('./data/cv_data.csv', index_col=0)
sess['LST_SESS_TIME'] = pd.to_datetime(sess['LST_SESS_TIME'])
print(sess.shape)
sess.head(2)

(293032, 18)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020,WEEKDAY,WEEKEND
0,000117b4880909a482cbfb7a65ddf0f71722298631,22,6,1,5,22,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 09:19:43.659,26.250000,0,1,7695.454545,1,0,0,0,1,0
1,000117b4880909a482cbfb7a65ddf0f71722301833,51,13,3,23,51,56c306ce1d710f24c64c604ce3dfc456c52d28ce4b62c0...,2024-07-30 10:29:11.286,44.677419,0,1,7695.454545,1,0,0,0,1,0


In [3]:
msg = pd.read_csv('./data/msg_indicate.csv', index_col = 0)
msg['STND_YMD'] = pd.to_datetime(msg['STND_YMD'])
print(msg.shape)
msg.head(2)

(4931532, 7)


,STND_YMD,INCS_NO,msg_length,max_discount,emoji_count,personalized,time_pressure
0,2024-10-28,f3f306c20e5caa497df4bbbd4732161eb63f65b92544b3...,76,64.0,1,1,1
1,2024-09-08,906522474e459ca44336c858e76b7f6cc4a8b76abba594...,49,33.0,2,0,0


In [4]:
cust_list = sess['INCS_NO'].unique()

In [5]:
new_msg = msg[msg['INCS_NO'].isin(cust_list)]
new_msg['msg_index'] = new_msg.index
new_msg = new_msg[(new_msg['STND_YMD'] >= pd.to_datetime('2024-06-01')) & (new_msg['STND_YMD'] < pd.to_datetime('2024-08-01'))]
new_msg

,STND_YMD,INCS_NO,msg_length,max_discount,emoji_count,personalized,time_pressure,msg_index
2,2024-06-21,a5028798964bfd1c9e1077155642b7196abca1d43d6ab6...,30,32.000000,0,0,0,2
26,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,273,40.000000,1,1,0,26
27,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,329,40.000000,1,1,0,27
28,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,291,40.000000,1,1,0,28
29,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,295,40.000000,1,1,0,29
...,...,...,...,...,...,...,...,...
5463200,2024-06-14,b491aa1ad5e874ae610c7931b40949c035cc6b4a9ee957...,40,50.000000,0,0,0,5463200
5463220,2024-07-12,380f002133125c673524d9eb310de1948ff5342a9c8fc4...,62,33.000000,0,0,0,5463220
5463229,2024-07-12,bbcc097782b6072d13605333820de38aa8f11fc9c8699a...,20,50.000000,0,0,0,5463229
5463239,2024-07-26,4dc13ea1a4a1613d8175252a0a3ae3794611f642bc6775...,62,33.333333,1,0,0,5463239


In [6]:
#gpt
new_msg["DATE_TIME"] = new_msg["STND_YMD"] + pd.Timedelta(hours=10)
new_msg.reset_index()
new_msg

,STND_YMD,INCS_NO,msg_length,max_discount,emoji_count,personalized,time_pressure,msg_index,DATE_TIME
2,2024-06-21,a5028798964bfd1c9e1077155642b7196abca1d43d6ab6...,30,32.000000,0,0,0,2,2024-06-21 10:00:00
26,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,273,40.000000,1,1,0,26,2024-07-29 10:00:00
27,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,329,40.000000,1,1,0,27,2024-07-29 10:00:00
28,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,291,40.000000,1,1,0,28,2024-07-29 10:00:00
29,2024-07-29,684734b317ac76a35202397a4cd6ecfbc23cfa38274e76...,295,40.000000,1,1,0,29,2024-07-29 10:00:00
...,...,...,...,...,...,...,...,...,...
5463200,2024-06-14,b491aa1ad5e874ae610c7931b40949c035cc6b4a9ee957...,40,50.000000,0,0,0,5463200,2024-06-14 10:00:00
5463220,2024-07-12,380f002133125c673524d9eb310de1948ff5342a9c8fc4...,62,33.000000,0,0,0,5463220,2024-07-12 10:00:00
5463229,2024-07-12,bbcc097782b6072d13605333820de38aa8f11fc9c8699a...,20,50.000000,0,0,0,5463229,2024-07-12 10:00:00
5463239,2024-07-26,4dc13ea1a4a1613d8175252a0a3ae3794611f642bc6775...,62,33.333333,1,0,0,5463239,2024-07-26 10:00:00


In [7]:
merged_df = pd.merge(
    new_msg[["msg_index","INCS_NO", "DATE_TIME"]],
    sess[["INCS_NO", "LST_SESS_TIME", "SESS_ID"]],
    on='INCS_NO',
    how='left'
)

In [8]:
# 차이를 일 단위로 계산
merged_df['date_diff'] = (merged_df['DATE_TIME'] - merged_df['LST_SESS_TIME']).dt.total_seconds() / (24 * 3600)
# -2일에서 2일 사이의 범위로 필터링
test_df = merged_df[merged_df['date_diff'] <= 1]
test_df.head(2)

,msg_index,INCS_NO,DATE_TIME,LST_SESS_TIME,SESS_ID,date_diff
0,2,a5028798964bfd1c9e1077155642b7196abca1d43d6ab6...,2024-06-21 10:00:00,2024-06-28 19:28:21.829,b689c491e744a66411c3d5ea1c574c5e1719570370,-7.394697
16,47,72db74633d53d603d376905eb946ef7448e4c5e801460e...,2024-07-12 10:00:00,2024-07-14 09:48:56.255,97A3754708664784B4F45C853C9E6ACC1720917359,-1.992318


In [9]:
def dv_coding(date_diff):
    if date_diff > 0:
        return 1
    else:
        return 0

test_df['y'] = test_df['date_diff'].apply(dv_coding)

In [10]:
test_df.y.value_counts()

y
0    1016133
1      31903
Name: count, dtype: int64

### new_msg, test_df, sess, cust 데이터 결합

In [19]:
final_df = test_df[['msg_index', 'INCS_NO', 'SESS_ID', 'y']]
final_df.head(2)

,msg_index,INCS_NO,SESS_ID,y
0,2,a5028798964bfd1c9e1077155642b7196abca1d43d6ab6...,b689c491e744a66411c3d5ea1c574c5e1719570370,0
16,47,72db74633d53d603d376905eb946ef7448e4c5e801460e...,97A3754708664784B4F45C853C9E6ACC1720917359,0


In [20]:
final_df = final_df.merge(new_msg, on='msg_index', how='left')
final_df = final_df.merge(sess, on='SESS_ID', how='left')
final_df.head(2)

,msg_index,INCS_NO_x,SESS_ID,y,STND_YMD,INCS_NO_y,msg_length,max_discount,emoji_count,personalized,...,AVG_DEPTH,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020,WEEKDAY,WEEKEND
0,2,a5028798964bfd1c9e1077155642b7196abca1d43d6ab6...,b689c491e744a66411c3d5ea1c574c5e1719570370,0,2024-06-21,a5028798964bfd1c9e1077155642b7196abca1d43d6ab6...,30,32.0,0,0,...,24.285714,1,1,21490.0,0,0,0,0,4,0
1,47,72db74633d53d603d376905eb946ef7448e4c5e801460e...,97A3754708664784B4F45C853C9E6ACC1720917359,0,2024-07-12,72db74633d53d603d376905eb946ef7448e4c5e801460e...,41,0.0,0,0,...,16.666667,0,1,26085.0,0,1,0,0,6,1


In [21]:
final_df.columns

Index(['msg_index', 'INCS_NO_x', 'SESS_ID', 'y', 'STND_YMD', 'INCS_NO_y',
       'msg_length', 'max_discount', 'emoji_count', 'personalized',
       'time_pressure', 'DATE_TIME', 'SRCH_EFRT', 'PRDV_CNT', 'CAT_CNT',
       'SAMGE_PAGE_CNT', 'EVNT_CNT', 'INCS_NO', 'LST_SESS_TIME', 'AVG_DEPTH',
       'day_off', 'SEX_CD', 'AVG_SAL_AMT', 'AGE_30', 'AGE_50', 'AGE_60',
       'AGE_1020', 'WEEKDAY', 'WEEKEND'],
      dtype='object')

In [22]:
final_df.drop(columns=['msg_index', 'INCS_NO_x', 'SESS_ID', 'STND_YMD', 'DATE_TIME', 'INCS_NO_y', 'INCS_NO', 'LST_SESS_TIME'], inplace=True)
final_df.head(2)

,y,msg_length,max_discount,emoji_count,personalized,time_pressure,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,...,AVG_DEPTH,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020,WEEKDAY,WEEKEND
0,0,30,32.0,0,0,0,6,2,0,2,...,24.285714,1,1,21490.0,0,0,0,0,4,0
1,0,41,0.0,0,0,0,15,4,0,4,...,16.666667,0,1,26085.0,0,1,0,0,6,1


In [16]:
final_df.to_csv('./data/final.csv')

In [17]:
len(final_df)

1048036